In [ ]:
from pathlib import Path
import sys, json, hashlib, io, contextlib, inspect
import numpy as np
import pandas as pd
from IPython.display import display, Image
import matplotlib
matplotlib.use('Agg')
from matplotlib import pyplot as plt
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'STATE_INPUTS.json').is_file()), None)
assert root is not None, 'Extract the complete STATE input bundle and open this notebook inside it.'
assets = root/'assets'
manifest = json.loads((root/'STATE_INPUTS.json').read_text())
def checked(relative):
    p = assets/relative
    assert hashlib.sha256(p.read_bytes()).hexdigest() == manifest[relative]['sha256'], relative
    return p
P = assets/'primary'
sys.path.insert(0, str(P/'scripts'))
import state_spectra_loop as base
base.ROOT=P; base.TARGET=checked('primary/target_model_results.csv'); base.EXT=P/'external'
raw = base.load_results()
assert raw.row_key.is_unique
print(f'{len(raw):,} scored perturbation/context/representation records. Individual cells are not the sample unit.')


In [ ]:
def compare_archive(actual, relative, keys, columns):
    expected = pd.read_csv(checked('primary/'+relative))
    a=actual.set_index(keys).sort_index(); b=expected.set_index(keys).sort_index()
    assert a.index.equals(b.index), 'Archived sample/level identities differ'
    for col in columns:
        np.testing.assert_allclose(a[col], b[col], rtol=1e-9, atol=1e-10, equal_nan=True, err_msg=col)
    print('Matched archived identities and values:', relative)

def plot_rows(df, level_col, metric='overlap_at_N', grouping=('spectra_subset','representation')):
    # New replay intervals use the original observation unit (perturbation within context).
    # They quantify sample resampling, not unseen-patient/site or fresh-inference uncertainty.
    rows=[]; rng=np.random.default_rng(7)
    for keys,g in df.groupby([*grouping,level_col],observed=True):
        v=pd.to_numeric(g[metric],errors='coerce').dropna().to_numpy(float)
        if not len(v):continue
        means=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(1000)])
        rows.append(dict(zip([*grouping,'level'],keys)) | {'n':len(v),'mean':v.mean(),'lo':np.quantile(means,.025),'hi':np.quantile(means,.975)})
    table=pd.DataFrame(rows); display(table)
    fig,ax=plt.subplots(figsize=(8,4.5),layout='constrained')
    levs=sorted(table.level.unique(),key=str)
    for keys,g in table.groupby(list(grouping),observed=True):
        g=g.assign(x=g.level.map({v:i for i,v in enumerate(levs)})).sort_values('x')
        ax.errorbar(g.x,g['mean'],yerr=np.vstack([g['mean']-g.lo,g.hi-g['mean']]),marker='o',capsize=2,label=str(keys),linewidth=1)
    ax.set_xticks(range(len(levs)),[str(v) for v in levs],rotation=25,ha='right');ax.set_ylabel(metric);ax.set_xlabel('Declared axis level');ax.legend(fontsize=7)
    ax.spines[['top','right']].set_visible(False)
    b=io.BytesIO();fig.savefig(b,format='png',dpi=120);plt.close(fig);display(Image(data=b.getvalue()))
    return table


In [ ]:
import tomllib
split_specs={}
for p in sorted((P/'external').glob('*.toml')):
    if p.stem.endswith('_zeroshot'):continue
    s=tomllib.loads(p.read_text()).get('fewshot',{}).get('replogle.'+p.stem,{})
    split_specs[p.stem]={k:{base.norm_symbol(v) for v in s.get(k,[])} for k in ('test','val')}
# The original audit infers its training reference from zero-shot gene identities,
# excluding each context's test/validation list. This is its recorded proxy, not a new assertion of training membership.
hgnc,alias=base.load_hgnc(); reactome=base.load_reactome(hgnc,alias)
features,coverage=base.add_replogle_features(raw,hgnc,alias,reactome,split_specs)
frozen,freeze=base.freeze_replogle_reactome_axis(features)
archived=pd.read_csv(checked('primary/spectra_ready_dataset/replogle_fewshot_reactome_support_examples.csv'),low_memory=False)
keys=['row_key','source_repo','source_path','perturbation']
a=frozen.set_index(keys).sort_index(); b=archived.set_index(keys).sort_index()
assert a.index.equals(b.index)
for col in ['reactome_pathway_count','reactome_max_jaccard_to_train','reactome_pathway_train_support_min','reactome_pathway_train_support_mean','reactome_pathway_train_support_log_mean','context_train_gene_count','symbol_family_candidate_train_count']:
    np.testing.assert_allclose(a[col],b[col],rtol=1e-10,atol=1e-10,equal_nan=True,err_msg=col)
print('Reactome and symbol-prefix features recomputed from annotations and split definitions; matched all 8,116 original records.')
display(frozen[['row_key','context','representation','perturbation','spectra_subset','reactome_pathway_train_support_log_mean']].head())


In [ ]:
import state_replogle_zeroshot_confirmation as z
z.base=base
# Run the original cross-protocol calculation in a separate output directory.
import tempfile,shutil
with tempfile.TemporaryDirectory() as td:
    run=Path(td);(run/'artifacts').mkdir();(run/'spectra_ready_dataset').mkdir()
    shutil.copy2(checked('primary/spectra_ready_dataset/state_replogle_reactome_pathway_log_train_support_v2_examples.csv'),run/'spectra_ready_dataset/state_replogle_reactome_pathway_log_train_support_v2_examples.csv')
    z.ROOT=run;z.ART=run/'artifacts';z.READY=run/'spectra_ready_dataset'
    with contextlib.redirect_stdout(io.StringIO()):z.main()
    data=pd.read_csv(z.READY/'replogle_zeroshot_reactome_support_confirmation_examples.csv',low_memory=False)
level='zeroshot_norm_support_level_from_fewshot_freeze'
feature='zeroshot_reactome_support_norm_log_mean'
curve,_=z.summarize(data,level,feature)
compare_archive(curve,'artifacts/spc_replogle_zeroshot_reactome_support_confirmation.csv',['spectra_subset','representation','level'],['n_rows','similarity_mean','overlap_at_N_mean'])
plot_rows(data,level)

In [ ]:
# These are official saved evaluation outputs. The archive does not contain the
# corresponding predicted expression vectors and ground truth matrices.
# Therefore overlap@N, DE recall and Pearson delta cannot be independently re-scored here.
assert np.allclose(data.overlap_at_N, raw.set_index('row_key').loc[data.row_key,'overlap_at_N'],equal_nan=True)
print('Every selected outcome links to the original scored row_key; fresh inference and metric re-scoring remain unavailable.')
